# 04 — Continuous batching and the serving runtime (Phase 4)

The first phase where the unit of measurement is a *request* rather than
a generation call, and where TTFT means queue + prefill rather than
prefill alone.

Ragged execution needs the paged cache, so Phase 3 is a prerequisite
here rather than an optimization.

**Budget:** ~2 minutes for the tests, ~20 minutes for the sweep.

In [ ]:
import os

# Set before torch is imported anywhere, and before any `!python` child
# process inherits the environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Two ways in. If you attached the repo as a Kaggle Dataset, point
# UPLOAD at it; otherwise it clones. A private repo needs a PAT in
# Add-ons -> Secrets as GITHUB_TOKEN.
REPO  = "https://github.com/Jash-Vora/LatentServe.git"
DEST  = Path("/kaggle/working/LatentServe")
UPLOAD = Path("/kaggle/input/latentserve")   # optional dataset fallback

if not DEST.exists():
    if UPLOAD.exists():
        !cp -r {UPLOAD} {DEST}
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass  # public repo, or you're pulling a fork
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        subprocess.run(["git", "clone", "--depth", "1", url, str(DEST)], check=True)

os.chdir(DEST)
sys.path.insert(0, str(DEST))
os.environ["PYTHONPATH"] = f"{DEST}:" + os.environ.get("PYTHONPATH", "")
print("cwd:", Path.cwd())
!git log --oneline -1

In [ ]:
# Do NOT install torch here. Kaggle ships a build matched to its driver;
# replacing it is the fastest way to lose an afternoon.
!pip install -q pydantic pyyaml 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
if (major, minor) < (4, 40):
    print("\n[!] pyproject requires transformers>=4.40. Run:  !pip install -q -U transformers")
else:
    print("transformers OK (LatentServe's own layer loop is version-independent "
          "by design, but model/qwen.py's HF cache walk needs >=4.40)")

In [ ]:
# Pull the checkpoint once, up front, so the download (~3 GB) never lands
# inside a timed benchmark. Later sessions re-download unless you save the
# HF cache as a Kaggle Dataset — worth doing once you're iterating daily.
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={cfg.num_hidden_layers} q_heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} "
      f"group={cfg.num_attention_heads // cfg.num_key_value_heads} "
      f"head_dim={getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads}")
_ = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
_ = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
del _
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("cached")

## 1. Gate 3 — serving must not change what any request receives

Four ragged requests generated alone, then through the engine at
concurrency 1, 2 and 4, demanding identical output ids. If concurrency
changes the tokens, no throughput number means anything.

In [ ]:
!python -m pytest tests/test_phase4_serving.py -v 2>&1 | tail -30

## 2. Continuous vs static batching

`--max-prompt 8192` clamps the workload's 64K tail (one 64K prefill is
~70 s and this experiment is about requests interacting, not about one
long prefill). `--max-output 256` keeps the 128/256 output mix intact —
static batching only loses when output lengths vary, so collapsing them
would hide the effect being measured.

In [ ]:
!python -m benchmarks.runners.phase4_serving \
    --config configs/phase4_serving.yaml --experiment batching \
    --workload mixed --num-requests 32 \
    --max-prompt 8192 --max-output 256 \
    --batch-sizes 1 4 8

## 3. Scheduler policies at fixed concurrency

FIFO, shortest-job-first, SJF with aging, and earliest-deadline. Watch
p99 TTFT, not just p50: a policy that improves the median by starving
its tail has made the server worse.

In [ ]:
!python -m benchmarks.runners.phase4_serving \
    --config configs/phase4_serving.yaml --experiment schedulers \
    --workload mixed --num-requests 32 \
    --max-prompt 8192 --max-output 256 --max-running 8

## 4. Check the predictions

In [ ]:
import pandas as pd

df = pd.read_json("results/raw/phase4_serving.jsonl", lines=True)
df = pd.concat([df.drop(columns=["extra"]), pd.json_normalize(df["extra"])], axis=1)

batching = df[df.system.str.contains("continuous|static")]
print(batching[["system", "batch_size", "output_tokens_per_s", "ttft_p50", "ttft_p95",
                "tpot_p50", "tpot_p99", "mean_batch_occupancy"]]
      .sort_values(["batch_size", "system"]).round(2).to_string(index=False))

for b in sorted(batching.batch_size.unique()):
    grp = batching[batching.batch_size == b]
    s = grp[grp.system.str.contains("static")]
    c = grp[grp.system.str.contains("continuous")]
    if len(s) and len(c):
        occ = s.mean_batch_occupancy.iloc[0] / b
        speedup = c.output_tokens_per_s.iloc[0] / s.output_tokens_per_s.iloc[0]
        print(f"\nbatch={b}: static occupancy {occ:.2f} of nominal -> P1 predicts "
              f"{1/occ:.2f}x, measured {speedup:.2f}x")

In [ ]:
sched = df[df.system.str.startswith("latentserve_sched_")].copy()
sched["policy"] = sched.system.str.replace("latentserve_sched_", "", regex=False)
print(sched[["policy", "output_tokens_per_s", "ttft_p50", "ttft_p95", "ttft_p99",
             "queue_p50", "tpot_p50", "tpot_p99",
             "scheduler_overhead_us_per_call", "runtime_overhead_pct"]]
      .round(2).to_string(index=False))
print("\nP2: p99/p50 TPOT ratio =",
      (sched.tpot_p99 / sched.tpot_p50).round(1).tolist(),
      "\n  (prefill blocks decode by design; large ratio is the argument for "
      "chunked mixed batching in Phase 17)")
print("P4: length_aware should beat fifo on ttft_p50 and lose on ttft_p99")

## 5. Before you close the session

Fill in the Gate 3 checklist in `docs/phase4.md` and mark each
prediction confirmed or falsified with its number. Then zip the results
— `results/raw/*.jsonl` is gitignored and exists only as notebook
output.

In [ ]:
!cd /kaggle/working/LatentServe && zip -qr /kaggle/working/results_phase4.zip results/raw
!ls -la /kaggle/working/*.zip